# 05 - Model Monitoring

**Azure ML MLOps Workshop | Block 3 (continued)**

### Why Model Monitoring?

Models degrade silently. The inspection comments from the field will evolve:
- New equipment types get introduced (new vocabulary)
- Inspection procedures change (different comment patterns)
- Seasonal or regional shifts in maintenance patterns

Without monitoring, you won't know your model's accuracy has dropped until
someone notices bad lead predictions months later.

Azure ML Model Monitoring provides:
- **Data drift detection** - Has the input distribution changed?
- **Prediction drift detection** - Has the model's output distribution changed?
- **Data quality signals** - Are we getting null values, unexpected formats?
- **Automated alerts** - Email/Teams when thresholds are breached

### What you'll do:
1. Understand monitoring signals for the lead classifier
2. Configure data drift monitoring
3. Configure prediction drift monitoring
4. Set up alerts and schedules

---

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<YOUR_SUBSCRIPTION_ID>"  # <<<< CHANGE THIS TO YOUR AZURE SUBSCRIPTION ID
RESOURCE_GROUP = "<YOUR_RESOURCE_GROUP>"  # <<<< CHANGE THIS TO YOUR RESOURCE GROUP (e.g., rg-aml-workshop-jd)
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"  # <<<< CHANGE THIS TO YOUR WORKSPACE NAME (e.g., aml-workshop-jd)

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
print(f"Connected to: {ml_client.workspace_name}")

## Step 1: Understand What We're Monitoring

For the Contoso lead classifier, the critical monitoring signals are:

| Signal | What Could Go Wrong | Business Impact |
|--------|--------------------|-----------------|
| **Data drift** | New equipment types, new terminology in comments | Model sees unfamiliar text, predictions become unreliable |
| **Prediction drift** | Model starts predicting more leads (or fewer) than training distribution | Sales team gets flooded with false leads, or misses real ones |
| **Data quality** | Null comments, encoding issues, truncated text | Model receives garbage input, outputs nonsense |

Let's first establish the training data baseline.

In [ ]:
import pandas as pd
import sys, os
sys.path.insert(0, os.path.abspath("../../src/track_a_text"))
from preprocess import load_and_clean_inspections

df = load_and_clean_inspections("../../data/inspections_dataset.csv")

print("Training data baseline:")
print(f"  Total samples: {len(df):,}")
print(f"  Lead rate: {df['label'].mean():.2%}")
print(f"  Avg comment length: {df['comment_clean'].str.len().mean():.1f} chars")
print(f"  Null rate: {df['comment_clean'].isnull().mean():.2%}")
print(f"  Confidence distribution:")
for conf, count in df['confidence'].value_counts().items():
    print(f"    {conf}: {count} ({count/len(df):.1%})")

## Step 2: Configure Model Monitoring

We set up a monitoring schedule that runs weekly, checking the deployed endpoint's
inference data against the training baseline.

In [ ]:
from azure.ai.ml import Input
from azure.ai.ml.entities import (
    MonitoringTarget,
    MonitorDefinition,
    MonitorSchedule,
    RecurrencePattern,
    RecurrenceTrigger,
    ServerlessSparkCompute,
    DataDriftSignal,
    PredictionDriftSignal,
    DataQualitySignal,
    ReferenceData,
    DataDriftMetricThreshold,
    DataQualityMetricThreshold,
    PredictionDriftMetricThreshold,
    NumericalDriftMetrics,
    CategoricalDriftMetrics,
    DataQualityMetricsNumerical,
    DataQualityMetricsCategorical,
    AlertNotification,
)

ENDPOINT_NAME = "contoso-lead-classifier"  # <<<< CHANGE THIS - MUST MATCH THE NAME YOU USED IN NOTEBOOK 04
DEPLOYMENT_NAME = "blue"

In [ ]:
# Reference data: the training dataset (v2)
reference_data = ReferenceData(
    input_data=Input(
        path="azureml:classified-inspections:2",
        type="uri_file",
    ),
    data_context="training",
)

# Monitoring target: the deployed endpoint
monitoring_target = MonitoringTarget(
    ml_task="classification",
    endpoint_deployment_id=f"azureml:{ENDPOINT_NAME}:{DEPLOYMENT_NAME}",
)

In [ ]:
# Signal 1: Data Drift
data_drift_signal = DataDriftSignal(
    reference_data=reference_data,
    metric_thresholds=DataDriftMetricThreshold(
        numerical=NumericalDriftMetrics(jensen_shannon_distance=0.25),
        categorical=CategoricalDriftMetrics(pearsons_chi_squared_test=0.3),
    ),
)

# Signal 2: Prediction Drift
prediction_drift_signal = PredictionDriftSignal(
    metric_thresholds=PredictionDriftMetricThreshold(
        categorical=CategoricalDriftMetrics(pearsons_chi_squared_test=0.3),
    ),
)

# Signal 3: Data Quality
data_quality_signal = DataQualitySignal(
    reference_data=reference_data,
    metric_thresholds=DataQualityMetricThreshold(
        numerical=DataQualityMetricsNumerical(null_value_rate=0.1),
        categorical=DataQualityMetricsCategorical(out_of_bounds_rate=0.1),
    ),
)

print("Monitoring signals configured:")
print("  1. Data drift (Jensen-Shannon: 0.25, Chi-squared: 0.3)")
print("  2. Prediction drift (Chi-squared: 0.3)")
print("  3. Data quality (null_value_rate: 0.1, out_of_bounds_rate: 0.1)")

In [ ]:
# Create the monitor definition
monitor_definition = MonitorDefinition(
    compute=ServerlessSparkCompute(
        instance_type="Standard_DS3_v2",
        runtime_version="3.3",
    ),
    monitoring_target=monitoring_target,
    monitoring_signals={
        "data_drift": data_drift_signal,
        "prediction_drift": prediction_drift_signal,
        "data_quality": data_quality_signal,
    },
    alert_notification=AlertNotification(
        emails=["your-email@example.com"],  # <<<< CHANGE THIS TO YOUR EMAIL ADDRESS
    ),
)

# Create the schedule (weekly on Mondays at 6 AM)
monitor_schedule = MonitorSchedule(
    name="contoso-lead-monitor",
    trigger=RecurrenceTrigger(
        frequency="week",
        interval=1,
        schedule=RecurrencePattern(hours=6, minutes=0, week_days=["Monday"]),
    ),
    create_monitor=monitor_definition,
)

created_monitor = ml_client.schedules.begin_create_or_update(monitor_schedule).result()
print(f"\nMonitor created: {created_monitor.name}")
print(f"  Schedule: Weekly on Mondays at 6:00 AM")
print(f"  Alert emails: <YOUR_EMAIL>")

## Step 3: Simulate What Drift Looks Like

To illustrate the concept, let's see what happens when production data
diverges from training data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Training distribution: ~25% leads, ~75% not leads
training_dist = [0.25, 0.75]

# Simulated production scenarios
scenarios = {
    "Normal (no drift)": [0.24, 0.76],
    "Mild drift": [0.35, 0.65],
    "Significant drift": [0.50, 0.50],
    "Severe drift": [0.70, 0.30],
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (name, dist) in enumerate(scenarios.items()):
    ax = axes[i]
    x = np.arange(2)
    width = 0.35
    ax.bar(x - width/2, training_dist, width, label="Training", color="#1f77b4")
    ax.bar(x + width/2, dist, width, label="Production", color="#ff7f0e")
    ax.set_xticks(x)
    ax.set_xticklabels(["Lead", "Not Lead"])
    ax.set_ylim(0, 1)
    ax.set_title(name)
    ax.legend(fontsize=8)

plt.suptitle("Prediction Drift Scenarios", fontsize=14)
plt.tight_layout()
plt.show()

## Step 4: The Retraining Trigger Workflow

When monitoring detects significant drift, the workflow is:

```
Monitor detects drift
  --> Alert sent (email / Teams webhook)
  --> Data team reviews new production data
  --> Collect and label new inspection comments
  --> Register as new data version (v3)
  --> Retrain using pipeline (Notebook 06)
  --> Register new model version (v2)
  --> Deploy as 'green' with canary traffic
  --> Validate metrics, full cutover
```

This cycle keeps the model accurate as Contoso's operations evolve.

In [ ]:
# List all monitoring schedules
print("Active monitoring schedules:")
for schedule in ml_client.schedules.list():
    print(f"  {schedule.name} | Status: {schedule.is_enabled} | Trigger: {schedule.trigger.frequency}")

## Go to Azure ML Studio Now

Navigate to **Monitoring** in the left navigation:

1. See the **contoso-lead-monitor** schedule
2. Once a run completes, explore:
   - **Data drift** charts showing feature distribution shifts
   - **Prediction drift** charts showing output distribution changes
   - **Data quality** metrics showing null rates and anomalies
3. Check the **alert configuration** for email notifications

---

## Multi-Region Monitoring Considerations

| Region | Monitoring Notes |
|--------|------------------|
| **Region 1** | Monitor for new sites introducing new equipment vocabulary. |
| **Region 2** | Separate baseline needed. Different equipment mix. |
| **Region 3** | Different equipment types. Different lead patterns. |

Each region should have its own monitoring schedule with region-specific baselines.

---

## Key Takeaways

| Concept | What We Did |
|---------|------------|
| **Data drift** | Detect when input data diverges from training distribution |
| **Prediction drift** | Detect when model outputs shift |
| **Data quality** | Catch nulls, encoding issues, unexpected formats |
| **Alerting** | Automated email notifications when thresholds breach |
| **Retraining loop** | Drift triggers data collection, retraining, safe redeployment |

---

**Next:** [06 - Pipeline Definition](./06_pipeline_definition.ipynb)